In [1]:
import polars as pl

INPUT_PATH  = "stgcn_dataset/node_features_X.parquet"
OUTPUT_PATH = "xgboost_dataset/xgb_tabular_24h.parquet"

In [7]:
LAG_BASE_COLS = [
    "demand",
    "revenue_total",
]

WEATHER_COLS = [
    "temperature",
    "wind_speed",
    "precipitation",
]

ROLLING_SUMMARY_COLS = [
    "rolling_tip_pct",
    "rolling_avg_fare",
    "rolling_peak_ratio",
]

CALENDAR_COLS = [
    "hour",
    "weekday",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "is_holiday",
]

GEO_COLS = [
    "zone_area_sqkm",
    "dist_to_center_km",
]

ID_COLS = [
    "time_bin",
    "LocationID",
]

LAGS = list(range(1, 13)) + [24] # hours

In [3]:
raw_df = pl.read_parquet(INPUT_PATH)

In [9]:
df = raw_df.filter(pl.col("time_bin").dt.year() >= 2023)
df = df.sort(["LocationID", "time_bin"])

lag_exprs = []
for col in LAG_BASE_COLS:
    for lag in LAGS:
        lag_exprs.append(
            pl.col(col)
            .shift(lag)
            .over("LocationID")
            .alias(f"{col}_lag_{lag}")
        )

target_exprs = [
    pl.col("demand").shift(-1).over("LocationID").alias("target_demand_t_plus_1"),
    pl.col("revenue_total").shift(-1).over("LocationID").alias("target_revenue_t_plus_1"),
]

df_final = df.select(
    *ID_COLS,
    *GEO_COLS,
    *CALENDAR_COLS,
    *ROLLING_SUMMARY_COLS,
    *WEATHER_COLS,
    *lag_exprs,
    *target_exprs,
)

df_final = df_final.drop_nulls()

In [11]:
df_final.write_parquet(OUTPUT_PATH)
print(f"\nSaved XGBoost dataset to {OUTPUT_PATH}")


Saved XGBoost dataset to xgboost_dataset/xgb_tabular_24h.parquet


In [10]:
df_final

time_bin,LocationID,zone_area_sqkm,dist_to_center_km,hour,weekday,hour_sin,hour_cos,weekday_sin,weekday_cos,is_holiday,rolling_tip_pct,rolling_avg_fare,rolling_peak_ratio,temperature,wind_speed,precipitation,demand_lag_1,demand_lag_2,demand_lag_3,demand_lag_4,demand_lag_5,demand_lag_6,demand_lag_7,demand_lag_8,demand_lag_9,demand_lag_10,demand_lag_11,demand_lag_12,demand_lag_24,revenue_total_lag_1,revenue_total_lag_2,revenue_total_lag_3,revenue_total_lag_4,revenue_total_lag_5,revenue_total_lag_6,revenue_total_lag_7,revenue_total_lag_8,revenue_total_lag_9,revenue_total_lag_10,revenue_total_lag_11,revenue_total_lag_12,revenue_total_lag_24,target_demand_t_plus_1,target_revenue_t_plus_1
datetime[μs],i64,f64,f64,i8,i8,f64,f64,f64,f64,i32,f32,f64,f64,f32,f32,f32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,u32,f32
2023-01-02 00:00:00,1,7.343009,17.611394,0,1,0.0,1.0,0.781831,0.62349,1,0.133321,93.197819,0.419014,10.6,1.5,0.0,0,0,1,4,2,1,3,4,8,6,4,3,0,0.0,0.0,132.199997,260.200012,216.0,99.599998,265.01001,404.200012,515.76001,692.799988,526.890015,522.079956,0.0,0,0.0
2023-01-02 01:00:00,1,7.343009,17.611394,1,1,0.258819,0.965926,0.781831,0.62349,1,0.133321,93.197819,0.419014,10.6,0.0,0.0,0,0,0,1,4,2,1,3,4,8,6,4,0,0.0,0.0,0.0,132.199997,260.200012,216.0,99.599998,265.01001,404.200012,515.76001,692.799988,526.890015,0.0,0,0.0
2023-01-02 02:00:00,1,7.343009,17.611394,2,1,0.5,0.866025,0.781831,0.62349,1,0.133321,93.197819,0.419014,10.6,0.0,0.0,0,0,0,0,1,4,2,1,3,4,8,6,0,0.0,0.0,0.0,0.0,132.199997,260.200012,216.0,99.599998,265.01001,404.200012,515.76001,692.799988,0.0,0,0.0
2023-01-02 03:00:00,1,7.343009,17.611394,3,1,0.707107,0.707107,0.781831,0.62349,1,0.133321,93.197819,0.419014,10.6,2.1,0.0,0,0,0,0,0,1,4,2,1,3,4,8,0,0.0,0.0,0.0,0.0,0.0,132.199997,260.200012,216.0,99.599998,265.01001,404.200012,515.76001,0.0,1,116.539993
2023-01-02 04:00:00,1,7.343009,17.611394,4,1,0.866025,0.5,0.781831,0.62349,1,0.133321,93.197819,0.419014,10.0,0.0,0.0,0,0,0,0,0,0,1,4,2,1,3,4,0,0.0,0.0,0.0,0.0,0.0,0.0,132.199997,260.200012,216.0,99.599998,265.01001,404.200012,0.0,2,250.119995
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-08-26 18:00:00,263,0.616541,3.626352,18,2,-1.0,-1.8370e-16,0.974928,-0.222521,0,0.15636,14.638287,0.974059,25.6,3.1,0.0,93,81,93,80,92,86,83,99,128,117,96,48,101,1711.790039,1119.97998,1478.450073,1299.079956,1426.179932,1305.220093,1583.040039,1967.390015,2280.120117,1945.949951,1671.109985,748.940002,1672.170044,76,1064.390015
2025-08-26 19:00:00,263,0.616541,3.626352,19,2,-0.965926,0.258819,0.974928,-0.222521,0,0.15636,14.638287,0.974059,24.4,3.1,0.0,106,93,81,93,80,92,86,83,99,128,117,96,71,1635.98999,1711.790039,1119.97998,1478.450073,1299.079956,1426.179932,1305.220093,1583.040039,1967.390015,2280.120117,1945.949951,1671.109985,1228.47998,80,1208.660034
2025-08-26 20:00:00,263,0.616541,3.626352,20,2,-0.866025,0.5,0.974928,-0.222521,0,0.15636,14.638287,0.974059,25.0,3.1,0.0,76,106,93,81,93,80,92,86,83,99,128,117,81,1064.390015,1635.98999,1711.790039,1119.97998,1478.450073,1299.079956,1426.179932,1305.220093,1583.040039,1967.390015,2280.120117,1945.949951,1196.150024,72,1230.530029
